# 01 — Introduction to Apache Spark

**Environment:** Ubuntu 22.04 · Python 3.12.3 · OpenJDK 17 · Spark 4.1.x · local PySpark.

Run cells in order. Each notebook can start a Spark session independently.

**Learning goal:** Distinguish driver, executors, partitions, transformations, and actions. In local mode the Spark driver and worker processes run on one computer, but the distributed programming concepts are the same.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'transactions.csv').is_file()), None)
assert ROOT is not None, 'Open the workshop folder in Jupyter before running this notebook.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from training_lib import create_spark, read_raw, typed_transactions, quality_condition, clean_transactions, completed_enriched, revenue_by_category
from pyspark.sql import functions as F
spark = create_spark('TG117-Workshop')
print('Spark:', spark.version, '| Python:', sys.version.split()[0], '| master:', spark.sparkContext.master)

### Create a distributed DataFrame

`local[2]` allows Spark to schedule work on two local threads. It is **not** a two-machine cluster.

In [ ]:
numbers = spark.range(1, 13, 1, 4)
print('Partitions:', numbers.rdd.getNumPartitions())
numbers.show()

### Observe per-partition distribution

`spark_partition_id()` is useful for teaching; partition allocation is an implementation detail and can change after shuffles.

In [ ]:
numbers.select('id', F.spark_partition_id().alias('partition_id')).orderBy('id').show(20)

### Transformation versus action

A transformation constructs a logical plan. `show()` and `count()` are actions that trigger computation.

In [ ]:
doubled = numbers.withColumn('double_id', F.col('id') * 2)  # transformation
print('A new DataFrame was defined.')
doubled.filter(F.col('double_id') > 10).show()  # action

**Check:** What is a partition? Why is `count()` an action? Would running `local[2]` turn your laptop into a distributed cluster?